# Swin geo context ablation

Loads the completed fold0 geo checkpoint and evaluates validation RMSE while zeroing context feature groups. No retraining.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/kaggle/working/solafune-nowcast')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/shionsuio/solafune-nowcast.git', str(REPO)], check=True)

sys.path.insert(0, str(REPO / 'src'))

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU', 'torch:', torch.__version__)

from evaluate_geo_context_ablation import run

DATA_ROOT = Path('/kaggle/input/solafune-dataset-v2')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('/kaggle/input/datasets/suioshion/solafune-dataset-v2')

NOTEBOOK_ROOT = Path('/kaggle/input/notebooks/suioshion/solafune-swin-temporal-geo')
if not NOTEBOOK_ROOT.exists():
    NOTEBOOK_ROOT = Path('/kaggle/input/solafune-swin-temporal-geo')

candidates = list(NOTEBOOK_ROOT.rglob('best_fold0.pth'))
print('NOTEBOOK_ROOT:', NOTEBOOK_ROOT)
print('checkpoint candidates:', candidates[:5])
assert candidates, 'best_fold0.pth not found in geo kernel output'
CHECKPOINT_ROOT = candidates[0].parent

LOCATION_METADATA = REPO / 'data' / 'location_coordinates_manual.csv'


class Args:
    root = '/kaggle/working'
    kaggle_input_root = str(DATA_ROOT)
    checkpoint_root = str(CHECKPOINT_ROOT)
    fold = 0
    modes = 'all'
    batch_size = 16
    workers = 2
    model_subdir = 'swin_v2_temporal_geo'
    location_metadata_path = str(LOCATION_METADATA)
    output = 'outputs/swin_v2_temporal_geo/context_ablation_fold0.csv'
    no_amp = True


summary_path = run(Args())

import pandas as pd
summary = pd.read_csv(summary_path).sort_values('validation_rmse')
display(summary)
